In [26]:
import astropy
import astroquery
from astroquery.esa.euclid import Euclid
import os
import pandas as pd

In [27]:
os.chdir("..")
df = pd.read_csv(".passwords")
os.chdir("euclid_generator/")

user = str(df["user"])
password = str(df["password"])
Euclid.login(user=user, password=password)

INFO: Login to Euclid TAP server: eas.esac.esa.int:443/tap-server/tap/ [astroquery.esa.euclid.core]
401 Error 401:
<!doctype html><html lang="en"><head><title>HTTP Status 401 – Unauthorized</title><style type="text/css">body {font-family:Tahoma,Arial,sans-serif;} h1, h2, h3, b {color:white;background-color:#525D76;} h1 {font-size:22px;} h2 {font-size:16px;} h3 {font-size:14px;} p {font-size:12px;} a {color:black;} .line {height:1px;background-color:#525D76;border:none;}</style></head><body><h1>HTTP Status 401 – Unauthorized</h1><hr class="line" /><p><b>Type</b> Status Report</p><p><b>Message</b> Bad Credentials</p><p><b>Description</b> The request has not been applied to the target resource because it lacks valid authentication credentials for that resource.</p><hr class="line" /><h3>Apache Tomcat/10.1.52</h3></body></html>


ERROR: Error logging in TAP server: Error 401:
<!doctype html><html lang="en"><head><title>HTTP Status 401 – Unauthorized</title><style type="text/css">body {font-family:Tahoma,Arial,sans-serif;} h1, h2, h3, b {color:white;background-color:#525D76;} h1 {font-size:22px;} h2 {font-size:16px;} h3 {font-size:14px;} p {font-size:12px;} a {color:black;} .line {height:1px;background-color:#525D76;border:none;}</style></head><body><h1>HTTP Status 401 – Unauthorized</h1><hr class="line" /><p><b>Type</b> Status Report</p><p><b>Message</b> Bad Credentials</p><p><b>Description</b> The request has not been applied to the target resource because it lacks valid authentication credentials for that resource.</p><hr class="line" /><h3>Apache Tomcat/10.1.52</h3></body></html> [astroquery.esa.euclid.core]


In [28]:
print(df.columns)

Index(['user', 'password'], dtype='str')


In [29]:
tables = Euclid.load_tables(only_names=True, include_shared_tables=True)
print(f'* Found {len(tables)} tables')
print(*(table.name for table in tables if "phz" in table.name), sep="\n")

INFO: Retrieving tables... [astroquery.utils.tap.core]
INFO: Parsing tables... [astroquery.utils.tap.core]
INFO: Done. [astroquery.utils.tap.core]
* Found 98 tables
catalogue.phz_classification
catalogue.phz_galaxy_sed
catalogue.phz_nir_physical_parameters
catalogue.phz_photo_z
catalogue.phz_physical_parameters
catalogue.phz_qso_physical_parameters
catalogue.phz_star_sed
catalogue.phz_star_template
q1.phz_catalogue
q1.phz_catalogue_l3
sedm.phz_catalogue
sedm.phz_catalogue_l3


In [ ]:
query = """
SELECT 
    mer.object_id,
    phz.phz_median,
    morph.sersic_sersic_vis_radius
FROM 
    catalogue.mer_catalogue AS mer
JOIN
    q1.phz_catalogue AS phz
    ON mer.object_id = phz.object_id
JOIN
    catalogue.mer_morphology AS morph
    ON mer.object_id = morph.object_id
WHERE
    mer.vis_det = 1
    AND phz.phz_median BETWEEN 0.1 AND 2.5
    AND morph.sersic_sersic_vis_radius > 0
"""

data = Euclid.launch_job(
    query=query,
    dump_to_file=True,
    # output_file="morphology_catalogue.csv"
)


ERROR: Query failed: 
SELECT 
    mer.object_id,
    phz.phz_median,
    morph.sersic_sersic_vis_radius
FROM 
    catalogue.mer_catalogue AS mer
JOIN
    q1.phz_catalogue AS phz
    ON mer.object_id = phz.object_id
JOIN
    catalogue.mer_morphology AS morph
    ON mer.object_id = morph.object_id
WHERE
    mer.vis_det = 1
    AND phz.phz_median BETWEEN 0.1 AND 2.5
    AND morph.sersic_sersic_vis_radius > 0
, 'str' object has no attribute 'read' [astroquery.esa.euclid.core]


In [ ]:
query = """
    SELECT mer.object_id
    FROM catalogue.mer_morphology AS mer
"""

data = Euclid.launch_job(
    query=query,
    dump_to_file=True,
    # output_file="morphology_catalogue.csv"
)

In [ ]:
query = """
    SELECT phz.phz_median
    FROM catalogue.phz_photo_z AS phz
"""

data = Euclid.launch_job(
    query=query,
    dump_to_file=True,
    # output_file="morphology_catalogue.csv"
)

In [33]:
query = """
SELECT 
    morph.object_id, phz.phz_median, morph.sersic_sersic_vis_radius, morph.concentration, morph.sersic_sersic_vis_axis_ratio, morph.sersic_sersic_vis_index
FROM 
    catalogue.mer_morphology AS morph
JOIN
    catalogue.phz_photo_z AS phz
    ON morph.object_id = phz.object_id
WHERE
    phz.phz_median IS NOT NULL
"""

data = Euclid.launch_job(
    query=query,
    dump_to_file=True,
    output_file="data/morphology_catalogue.csv"
)